# F1-scientific-python — Practice p07 — Solution

In [ ]:
import numpy as np

SEED = 20260804
rng = np.random.default_rng(SEED)

In [ ]:
# Given: WORKING loop code. Run it, read it, then beat it.
SEED = 20260804
rng = np.random.default_rng(SEED)
data = rng.integers(0, 100, size=(200, 6)).astype(np.float64)   # 200 rows x 6 columns

n_rows, n_cols = data.shape

col_mean_loop = []
col_max_loop = []
col_span_loop = []
high_count_loop = []
for j in range(n_cols):
    total = 0.0
    biggest = data[0, j]
    smallest = data[0, j]
    high = 0
    for i in range(n_rows):
        v = data[i, j]
        total = total + v
        if v > biggest:
            biggest = v
        if v < smallest:
            smallest = v
        if v >= 90:
            high = high + 1
    col_mean_loop.append(total / n_rows)
    col_max_loop.append(biggest)
    col_span_loop.append(biggest - smallest)
    high_count_loop.append(high)

col_mean_loop = np.array(col_mean_loop)
col_max_loop = np.array(col_max_loop)
col_span_loop = np.array(col_span_loop)
high_count_loop = np.array(high_count_loop)

row_top_loop = []
for i in range(n_rows):
    best = data[i, 0]
    for j in range(n_cols):
        if data[i, j] > best:
            best = data[i, j]
    row_top_loop.append(best)
row_top_loop = np.array(row_top_loop)

print(col_mean_loop.round(2))
print(high_count_loop)

**Task A — per-column statistics, loop-free.** In one line each, compute:

- **`col_mean`** — average of each column (matches `col_mean_loop`)
- **`col_max`** — largest value in each column (matches `col_max_loop`)
- **`col_span`** — largest minus smallest per column (matches `col_span_loop`)
- **`high_count`** — how many values in each column are ≥ 90 (matches
  `high_count_loop`)

Each result must have shape `(6,)` — which axis disappears?

In [ ]:
col_mean = data.mean(axis=0)
col_max = data.max(axis=0)
col_span = data.max(axis=0) - data.min(axis=0)
high_count = (data >= 90).sum(axis=0)
print(col_mean.round(2))
print(high_count)

Per-column summaries collapse the row axis, so every aggregation uses `axis=0`. The 30-line double loop becomes four one-liners; the running total, the biggest/smallest trackers, and the `if v >= 90` counter map to `mean`, `max`/`min`, and a mask's `.sum()` respectively.

**Task B — per-row statistic, loop-free.** Compute **`row_top`**, the largest
value in each row (matches `row_top_loop`, shape `(200,)`).

In [ ]:
row_top = data.max(axis=1)
print(row_top[:10])

One value per row means the column axis collapses: `axis=1`.

**Task C — prove agreement.** Set **`all_match`** to `True` only if all five
of your arrays agree with their loop versions (use `np.allclose` for the
decimal results and `np.array_equal` for `high_count`).

In [ ]:
all_match = (
    np.allclose(col_mean, col_mean_loop, atol=1e-9, rtol=0)
    and np.allclose(col_max, col_max_loop, atol=1e-9, rtol=0)
    and np.allclose(col_span, col_span_loop, atol=1e-9, rtol=0)
    and np.array_equal(high_count, high_count_loop)
    and np.allclose(row_top, row_top_loop, atol=1e-9, rtol=0)
)
print("all five agree:", all_match)

Always verify a rewrite against the original: `np.allclose` tolerates tiny decimal rounding differences (the loop adds numbers one at a time, in a different order than `mean`), while exact integer counts can use `np.array_equal`.

**Task D — one step further.** Using broadcasting with your `col_mean`
(shape `(6,)` against `data`'s `(200, 6)`), compute **`centered`** — `data`
with each column's average subtracted from that column. Check that
`centered.mean(axis=0)` is (essentially) all zeros. No loops, naturally.

In [ ]:
centered = data - col_mean
print(centered.shape)
print(centered.mean(axis=0).round(12))

`(200, 6)` against `(6,)` lines the 6s up from the right, so `col_mean` is subtracted from every row — each column loses its own average, making the new per-column averages zero (up to tiny rounding).

### Answer check

In [ ]:
assert col_mean.shape == (6,) and np.allclose(col_mean, col_mean_loop, atol=1e-9, rtol=0)
assert col_max.shape == (6,) and np.allclose(col_max, col_max_loop, atol=1e-9, rtol=0)
assert col_span.shape == (6,) and np.allclose(col_span, col_span_loop, atol=1e-9, rtol=0)
assert high_count.shape == (6,) and np.array_equal(high_count, high_count_loop)
assert row_top.shape == (200,) and np.allclose(row_top, row_top_loop, atol=1e-9, rtol=0)
assert all_match is True
assert centered.shape == (200, 6)
assert np.allclose(centered.mean(axis=0), 0.0, atol=1e-9, rtol=0)
assert np.allclose(centered, data - data.mean(axis=0), atol=1e-9, rtol=0)
print("all checks passed")